# Official T2I-CompBench: 1,000-image seed-13 subset, full SD3/T5, dual GPU

This notebook is a new, T2I-only evaluation and does not modify or reuse outputs from the earlier combined T2I-CompBench/GenEval notebook.

It evaluates **Our Method (repository SPFC)**, **standard/base CFG**, and **Rectified CFG++ from the pinned official repository** on exactly the same SD3 Medium checkpoint, scheduler, 1,024×1,024 resolution, 28 steps, guidance scale 4.5, prompts, negative prompt, and generation seeds. The requested subset is 100 prompts selected with Python's `random.Random(13)`, with 25 prompts from each supported section (color, texture, spatial, shape). Seed 13 is used only for subset selection; following the official T2I-CompBench generator and the official Rectified-CFG++ default, ten images per prompt use seeds 42–51. This produces exactly **1,000 images per method** and **3,000 images total**.

The full SD3 text stack is used: CLIP-L, CLIP-G, and T5-XXL. T5-XXL does not fit on either local GPU together with SD3 inference, so the notebook first runs the unmodified `StableDiffusion3Pipeline.encode_prompt` with T5 sharded across both GPUs, caches each prompt with singleton batch topology, releases T5, and then passes those same cached tensors to every method. This exactly matches the per-prompt full-T5 SD3 conditioning path. The cache requires both an exact safetensors round-trip and bitwise equality against singleton CFG encoding. Generation then runs concurrently: GPU 0 handles Our Method, while GPU 1 handles base CFG followed by official Rectified CFG++. Per-device microbatch size is one because the smaller card has 8 GB VRAM; the two concurrent workers provide an effective global batch of two without lowering resolution or changing guidance.

Scoring calls the pinned official T2I-CompBench BLIP-VQA implementation for color/texture/shape and UniDet implementation for spatial relationships. This is a deterministic subset study, not the benchmark's full 300-prompts-per-section protocol, so its scores must be reported as **seed-13 subset scores** and must not be compared directly with full-set leaderboard values. GenEval and FID are deliberately absent per the updated scope.

The expensive stages are resumable. Valid image/JSON pairs and valid embedding-cache entries are not overwritten. All artifacts go to `outputs/t2i_compbench_1000_seed13_full_t5_dual_gpu` unless `AIM_FLOW_T2I1000_ROOT` is set.


In [2]:
# Install the generation environment and clone/verify official repositories.
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src" / "aim_flow").is_dir():
            return candidate.resolve()
    raise FileNotFoundError("Launch this notebook from inside the aim-flow repository.")


REPO_ROOT = find_repo_root(Path.cwd())
os.chdir(REPO_ROOT)
RUN_INSTALLS = os.environ.get("AIM_FLOW_SKIP_INSTALLS", "0") != "1"

PINS = {
    "rectified_cfgpp": (
        "https://github.com/shreshthsaini/Rectified-CFGpp.git",
        "3c838882f7b2cdc2e1c3e5785468cdcb1fb27190",
    ),
    "t2i_compbench": (
        "https://github.com/Karine-Huang/T2I-CompBench.git",
        "1b7094991a57f3c22abdd4f6e8ba6c1a15517073",
    ),
    "openai_clip": (
        "https://github.com/openai/CLIP.git",
        "d05afc436d78f1c48dc0dbf8e5980a9d471f35f6",
    ),
    "detectron2": (
        "https://github.com/facebookresearch/detectron2.git",
        "5aeb252b194b93dc2879b4ac34bc51a31b5aee13",
    ),
}
GENERATION_PACKAGES = [
    "torch==2.10.0", "torchvision==0.25.0", "diffusers==0.30.2",
    "transformers==4.49.0", "accelerate==1.13.0", "huggingface-hub==0.29.3",
    "safetensors==0.7.0", "sentencepiece==0.2.1", "protobuf==5.29.3",
    "numpy==2.4.6", "Pillow==12.1.1", "pandas==3.0.0", "tqdm==4.67.1",
    "PyYAML==6.0.3", "psutil==7.0.0", "nbformat==5.10.4",
]


def run(command, *, cwd: Path | None = None, env=None, capture: bool = False):
    command = [str(x) for x in command]
    print("+", " ".join(command))
    return subprocess.run(
        command, cwd=str(cwd or REPO_ROOT), env=env, check=True,
        text=True, capture_output=capture,
    )


if RUN_INSTALLS:
    run([sys.executable, "-m", "pip", "install", "--upgrade", "pip==24.3.1", "setuptools==75.6.0", "wheel==0.45.1"])
    run([sys.executable, "-m", "pip", "install", *GENERATION_PACKAGES])
    run([sys.executable, "-m", "pip", "install", "--no-deps", "-e", str(REPO_ROOT)])

EXTERNAL = REPO_ROOT / "external"
REPO_PATHS = {
    "rectified_cfgpp": EXTERNAL / "Rectified-CFGpp",
    "t2i_compbench": EXTERNAL / "T2I-CompBench",
}


def git_output(repo: Path, *args: str) -> str:
    return subprocess.check_output(["git", "-C", str(repo), *args], text=True).strip()


def ensure_pinned_checkout(name: str, destination: Path) -> Path:
    url, commit = PINS[name]
    if not destination.exists():
        destination.parent.mkdir(parents=True, exist_ok=True)
        run(["git", "clone", "--filter=blob:none", url, str(destination)])
    if not (destination / ".git").is_dir():
        raise RuntimeError(f"{destination} exists but is not a git checkout")
    if git_output(destination, "status", "--porcelain", "--untracked-files=no"):
        raise RuntimeError(f"Refusing to change dirty official checkout: {destination}")
    if git_output(destination, "rev-parse", "HEAD") != commit:
        run(["git", "fetch", "origin", commit], cwd=destination)
        run(["git", "checkout", "--detach", commit], cwd=destination)
    actual = git_output(destination, "rev-parse", "HEAD")
    if actual != commit:
        raise RuntimeError(f"Pin mismatch for {name}: expected {commit}, found {actual}")
    return destination


for repo_name, repo_path in REPO_PATHS.items():
    ensure_pinned_checkout(repo_name, repo_path)

print({name: {"path": str(path), "commit": git_output(path, "rev-parse", "HEAD")} for name, path in REPO_PATHS.items()})


+ /media/fezan/ASi/DVLM/steering/aim-flow/.venv/aim-flow/bin/python -m pip install --upgrade pip==24.3.1 setuptools==75.6.0 wheel==0.45.1



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


+ /media/fezan/ASi/DVLM/steering/aim-flow/.venv/aim-flow/bin/python -m pip install torch==2.10.0 torchvision==0.25.0 diffusers==0.30.2 transformers==4.49.0 accelerate==1.13.0 huggingface-hub==0.29.3 safetensors==0.7.0 sentencepiece==0.2.1 protobuf==5.29.3 numpy==2.4.6 Pillow==12.1.1 pandas==3.0.0 tqdm==4.67.1 PyYAML==6.0.3 psutil==7.0.0 nbformat==5.10.4



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


+ /media/fezan/ASi/DVLM/steering/aim-flow/.venv/aim-flow/bin/python -m pip install --no-deps -e /media/fezan/ASi/DVLM/steering/aim-flow
Obtaining file:///media/fezan/ASi/DVLM/steering/aim-flow
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for aim-flow (pyproject.toml): started
  Building editable for aim-flow (pyproject.toml): finished with status 'done'
  Created wheel for aim-flow: filename=aim_flow-0.1.0-0.editable-py3-none-any.whl size=11533 sha256=6c8a9f647d5c0747bc46ff7afaf650e215069a381722eb28644de070161968b5
  Stored


[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [3]:
# Lock the protocol, verify model access/full T5 files, and inventory this PC.
import gc
import hashlib
import importlib.metadata
import json
import math
import platform
import random
import shutil
import time
import urllib.request
from dataclasses import asdict, dataclass
from typing import Any, Iterable

import numpy as np
import pandas as pd
import torch
from PIL import Image
from huggingface_hub import HfFolder, model_info, snapshot_download
from tqdm.auto import tqdm

sys.path.insert(0, str(REPO_ROOT / "src"))

from aim_flow.eval_bench.constants import DEFAULT_SPFC_SCHEDULE_1_INDEXED


@dataclass(frozen=True)
class Protocol:
    model_id: str = "stabilityai/stable-diffusion-3-medium-diffusers"
    model_revision: str = "ea42f8cef0f178587cf766dc8129abd379c90671"
    scheduler: str = "checkpoint FlowMatchEulerDiscreteScheduler"
    dtype: str = "float16"
    full_text_encoders: tuple[str, ...] = ("CLIP-L", "CLIP-G", "T5-XXL")
    max_sequence_length: int = 256
    t5_cache_batch_size: int = 1
    height: int = 1024
    width: int = 1024
    num_inference_steps: int = 28
    guidance_scale: float = 4.5
    negative_prompt: str = ""
    subset_prompts: int = 100
    prompts_per_category: int = 25
    selection_seed: int = 13
    samples_per_prompt: int = 10
    generation_seed_start: int = 42
    rectified_sigma_noise: float = 0.005
    spfc_aggregation_steps_1_indexed: tuple[int, ...] = (1, 2, 3, 5, 6, 7, 8, 9, 10, 12, 14, 16, 19, 21, 23, 28)
    per_device_microbatch: int = 1
    physical_gpu_our_method: int = 0
    physical_gpu_baselines: int = 1


PROTOCOL = Protocol()
METHODS = ("our_method", "base_cfg", "rectified_cfgpp")
METHOD_LABELS = {"our_method": "Our Method", "base_cfg": "Base CFG", "rectified_cfgpp": "Rectified CFG++"}
T2I_CATEGORIES = ("color", "texture", "spatial", "shape")
T2I_SEEDS = tuple(PROTOCOL.generation_seed_start + i for i in range(PROTOCOL.samples_per_prompt))
ARTIFACT_ROOT = Path(os.environ.get(
    "AIM_FLOW_T2I1000_ROOT",
    REPO_ROOT / "outputs" / "t2i_compbench_1000_seed13_full_t5_dual_gpu",
)).resolve()
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)


def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def sha256_file(path: Path, chunk_size: int = 1 << 20) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def canonical_hash(value: Any) -> str:
    return sha256_bytes(json.dumps(value, sort_keys=True, separators=(",", ":")).encode("utf-8"))


def write_json(value: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    os.replace(temporary, path)


def remap_schedule(schedule: Iterable[int], source_steps: int, target_steps: int) -> tuple[int, ...]:
    mapped = tuple(round((step - 1) * (target_steps - 1) / (source_steps - 1)) + 1 for step in schedule)
    if tuple(sorted(set(mapped))) != mapped:
        raise RuntimeError(f"Schedule remapping collapsed or reordered steps: {mapped}")
    return mapped


mapped_schedule = remap_schedule(tuple(DEFAULT_SPFC_SCHEDULE_1_INDEXED), 24, PROTOCOL.num_inference_steps)
if mapped_schedule != PROTOCOL.spfc_aggregation_steps_1_indexed:
    raise RuntimeError(f"SPFC 24→28 schedule mismatch: {mapped_schedule}")

RECTIFIED_PIPELINE_SHA256 = "fd83394f3a9aefbb2b9a706c785aab3dc5251d9b05a8022c5053f37704ef113a"
rectified_pipeline = REPO_PATHS["rectified_cfgpp"] / "rect-cfg-SD3-pipeline" / "pipeline.py"
if sha256_file(rectified_pipeline) != RECTIFIED_PIPELINE_SHA256:
    raise RuntimeError("Pinned official Rectified-CFG++ pipeline hash mismatch")

if not torch.cuda.is_available() or torch.cuda.device_count() < 2:
    raise RuntimeError("This requested protocol requires two CUDA GPUs; fewer than two are visible.")
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN") or HfFolder.get_token()
if not HF_TOKEN:
    raise RuntimeError("Set HF_TOKEN (or run huggingface-cli login) with accepted SD3 Medium access. The token is never saved.")
try:
    info = model_info(PROTOCOL.model_id, revision=PROTOCOL.model_revision, token=HF_TOKEN)
    if info.sha != PROTOCOL.model_revision:
        raise RuntimeError(f"Resolved revision {info.sha} != {PROTOCOL.model_revision}")
    MODEL_SNAPSHOT = Path(snapshot_download(
        PROTOCOL.model_id, revision=PROTOCOL.model_revision, token=HF_TOKEN,
    )).resolve()
except Exception as exc:
    raise RuntimeError(f"Cannot access/download the pinned gated SD3 Medium checkpoint: {exc}") from exc

required_model_paths = [
    "transformer", "vae", "text_encoder", "text_encoder_2", "text_encoder_3",
    "tokenizer", "tokenizer_2", "tokenizer_3", "scheduler",
]
missing = [name for name in required_model_paths if not (MODEL_SNAPSHOT / name).exists()]
if missing:
    raise FileNotFoundError(f"The full SD3 snapshot is incomplete; missing components: {missing}")

PROTOCOL_HASH = canonical_hash(asdict(PROTOCOL))
hardware = {
    "platform": platform.platform(), "python": sys.version,
    "torch": torch.__version__, "cuda_runtime": torch.version.cuda,
    "gpus": [
        {"physical_index": i, "name": torch.cuda.get_device_name(i), "memory_bytes": torch.cuda.get_device_properties(i).total_memory}
        for i in range(torch.cuda.device_count())
    ],
}
protocol_record = {
    "protocol": asdict(PROTOCOL), "protocol_hash": PROTOCOL_HASH,
    "model_snapshot": str(MODEL_SNAPSHOT), "hardware": hardware,
    "repository_pins": {name: {"url": PINS[name][0], "commit": PINS[name][1]} for name in ("rectified_cfgpp", "t2i_compbench")},
    "rectified_pipeline_sha256": RECTIFIED_PIPELINE_SHA256,
    "parallel_assignment": {"gpu_0": ["our_method"], "gpu_1": ["base_cfg", "rectified_cfgpp"]},
    "conditioning_topology": "Full CLIP-L + CLIP-G + T5-XXL encode_prompt cache, then identical cached tensors supplied to all methods",
}
write_json(protocol_record, ARTIFACT_ROOT / "protocol.json")
print(json.dumps(protocol_record, indent=2))


/media/fezan/ASi/DVLM/steering/aim-flow/.venv/aim-flow/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 38 files: 100%|██████████| 38/38 [00:00<00:00, 43895.22it/s]

{
  "protocol": {
    "model_id": "stabilityai/stable-diffusion-3-medium-diffusers",
    "model_revision": "ea42f8cef0f178587cf766dc8129abd379c90671",
    "scheduler": "checkpoint FlowMatchEulerDiscreteScheduler",
    "dtype": "float16",
    "full_text_encoders": [
      "CLIP-L",
      "CLIP-G",
      "T5-XXL"
    ],
    "max_sequence_length": 256,
    "t5_cache_batch_size": 1,
    "height": 1024,
    "width": 1024,
    "num_inference_steps": 28,
    "guidance_scale": 4.5,
    "negative_prompt": "",
    "subset_prompts": 100,
    "prompts_per_category": 25,
    "selection_seed": 13,
    "samples_per_prompt": 10,
    "generation_seed_start": 42,
    "rectified_sigma_noise": 0.005,
    "spfc_aggregation_steps_1_indexed": [
      1,
      2,
      3,
      5,
      6,
      7,
      8,
      9,
      10,
      12,
      14,
      16,
      19,
      21,
      23,
      28
    ],
    "per_device_microbatch": 1,
    "physical_gpu_our_method": 0,
    "physical_gpu_baselines": 1
  },
  "prot

In [4]:
# Rebuild the balanced seed-13 subset from the official files and expand it to 1,000 tasks.
from collections import Counter

from aim_flow.eval_bench.prompt_sources import build_t2i_compbench_manifest
from aim_flow.eval_bench.schemas import DecompositionManifest, PromptManifest

T2I_REPO = REPO_PATHS["t2i_compbench"]
OFFICIAL_DATASET = T2I_REPO / "examples" / "dataset"
CHECKED_PROMPT_MANIFEST = REPO_ROOT / "configs" / "t2i_compbench_100_seed13.json"
CHECKED_DECOMPOSITIONS = REPO_ROOT / "configs" / "t2i_compbench_100_seed13_spfc.json"
EXPECTED_MANIFEST_SHA256 = "f9cf97555168f7ef825e841b8db717c4c7b12e4ef36a1aa75678bd10445cdf9e"
EXPECTED_DECOMPOSITION_SHA256 = "9c74731a1f424b567b8910287dbaa197045c8605d89e917bcb3ce7916444cf16"
OFFICIAL_PROMPT_HASHES = {
    "color": "1634259756dbc77d13093d907d414480080ec9790a8c97ad09efaea7b2534f2d",
    "texture": "fbb5363515b4a28009e360afaaaf389d2dd2289e50dbb0c6ef4ea0c7fe4f3d4f",
    "spatial": "8707a1d7e42ce3002d95363cf55f74bd043c843ae27108f5125aaaa7840ca988",
    "shape": "37e1a276906c7ea9516cbd7ac8501be896006c0c17ca7344f260562834455bae",
}
if sha256_file(CHECKED_PROMPT_MANIFEST) != EXPECTED_MANIFEST_SHA256:
    raise RuntimeError("Checked seed-13 prompt manifest hash mismatch")
if sha256_file(CHECKED_DECOMPOSITIONS) != EXPECTED_DECOMPOSITION_SHA256:
    raise RuntimeError("Checked SPFC decomposition manifest hash mismatch")
for category, expected_hash in OFFICIAL_PROMPT_HASHES.items():
    source = OFFICIAL_DATASET / f"{category}_val.txt"
    if sha256_file(source) != expected_hash:
        raise RuntimeError(f"Pinned official prompt file hash mismatch: {source}")

rebuilt = build_t2i_compbench_manifest(
    subset_size=PROTOCOL.subset_prompts,
    seed=PROTOCOL.selection_seed,
    dataset_root=OFFICIAL_DATASET,
)
checked = PromptManifest.load(CHECKED_PROMPT_MANIFEST)
if rebuilt.to_dict() != checked.to_dict():
    raise RuntimeError("Fresh random.Random(13) selection differs from the checked manifest")
counts = Counter(sample.category for sample in rebuilt.samples)
if counts != Counter({category: PROTOCOL.prompts_per_category for category in T2I_CATEGORIES}):
    raise RuntimeError(f"Subset is not balanced 25/category: {counts}")

decompositions = DecompositionManifest.load(CHECKED_DECOMPOSITIONS)
decomp_by_id = decompositions.item_by_id()
if set(decomp_by_id) != {sample.id for sample in rebuilt.samples}:
    raise RuntimeError("SPFC decomposition IDs do not exactly match the selected prompt IDs")
for sample in rebuilt.samples:
    item = decomp_by_id[sample.id]
    if item.target_prompt != sample.prompt:
        raise RuntimeError(f"Target/decomposition mismatch: {sample.id}")
    if "_" in sample.prompt or "/" in sample.prompt or "\\" in sample.prompt:
        raise ValueError(f"Prompt is incompatible with the official scorer filename parser: {sample.prompt!r}")

tasks = []
for sample in rebuilt.samples:
    original_index = int(sample.metadata["original_index"])
    category_rank = int(sample.id.rsplit("_", 1)[1])
    if not 0 <= category_rank < PROTOCOL.prompts_per_category:
        raise RuntimeError(f"Invalid category-local selected rank: {sample.id}")
    for sample_index, seed in enumerate(T2I_SEEDS):
        tasks.append({
            "subset_sample_id": sample.id,
            "category": sample.category,
            "prompt": sample.prompt,
            "official_prompt_index": original_index,
            "selected_category_rank": category_rank,
            "sample_index": sample_index,
            "question_id": category_rank * PROTOCOL.samples_per_prompt + sample_index,
            "seed": seed,
        })
if len(tasks) != 1000 or len({(x["category"], x["question_id"]) for x in tasks}) != 1000:
    raise RuntimeError("Expected exactly 1,000 unique T2I generation tasks")

MANIFEST_DIR = ARTIFACT_ROOT / "manifests"
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
rebuilt.save(MANIFEST_DIR / "selected_prompts.json")
decompositions.save(MANIFEST_DIR / "spfc_decompositions.json")
TASK_MANIFEST = MANIFEST_DIR / "generation_tasks.jsonl"
TASK_MANIFEST.write_text("\n".join(json.dumps(task, sort_keys=True) for task in tasks) + "\n", encoding="utf-8")
for category in T2I_CATEGORIES:
    selected = [sample.prompt for sample in rebuilt.samples if sample.category == category]
    (MANIFEST_DIR / f"{category}_selected_seed13.txt").write_text("\n".join(selected) + "\n", encoding="utf-8")

subset_audit = {
    "algorithm": "random.Random(13).sample independently per category; sampled indices sorted into official order",
    "selection_seed": 13, "counts": dict(counts), "task_count": len(tasks),
    "generation_seeds": list(T2I_SEEDS),
    "official_question_id_rule": "selected category-local prompt rank * 10 + sample_index (contiguous 000000..000249 per category)",
    "selected_manifest_sha256": sha256_file(MANIFEST_DIR / "selected_prompts.json"),
    "task_manifest_sha256": sha256_file(TASK_MANIFEST),
    "source_prompt_hashes": OFFICIAL_PROMPT_HASHES,
}
write_json(subset_audit, MANIFEST_DIR / "subset_audit.json")
print(json.dumps(subset_audit, indent=2))


{
  "algorithm": "random.Random(13).sample independently per category; sampled indices sorted into official order",
  "selection_seed": 13,
  "counts": {
    "color": 25,
    "shape": 25,
    "texture": 25,
    "spatial": 25
  },
  "task_count": 1000,
  "generation_seeds": [
    42,
    43,
    44,
    45,
    46,
    47,
    48,
    49,
    50,
    51
  ],
  "official_question_id_rule": "selected category-local prompt rank * 10 + sample_index (contiguous 000000..000249 per category)",
  "selected_manifest_sha256": "f9cf97555168f7ef825e841b8db717c4c7b12e4ef36a1aa75678bd10445cdf9e",
  "task_manifest_sha256": "f658b66cec77f4720628be491d24bf1b56e17665573a26f87931141c1710f179",
  "source_prompt_hashes": {
    "color": "1634259756dbc77d13093d907d414480080ec9790a8c97ad09efaea7b2534f2d",
    "texture": "fbb5363515b4a28009e360afaaaf389d2dd2289e50dbb0c6ef4ea0c7fe4f3d4f",
    "spatial": "8707a1d7e42ce3002d95363cf55f74bd043c843ae27108f5125aaaa7840ca988",
    "shape": "37e1a276906c7ea9516cbd7ac850

In [5]:
# Encode every text once with the full, unmodified SD3 text stack. T5-XXL is sharded over both GPUs.
from safetensors import safe_open
from safetensors.torch import load_file as load_safetensors, save_file as save_safetensors

if PROTOCOL.t5_cache_batch_size != 1:
    raise RuntimeError("Stale notebook state detected: rerun the protocol cell so singleton T5 caching is active.")

EMBEDDING_ROOT = ARTIFACT_ROOT / "full_t5_prompt_cache"
EMBEDDING_ROOT.mkdir(parents=True, exist_ok=True)
EMBEDDING_INDEX = EMBEDDING_ROOT / "index.json"

CACHE_IDENTITY = {
    "schema": "sd3_full_text_condition_v1",
    "model_id": PROTOCOL.model_id,
    "model_revision": PROTOCOL.model_revision,
    "diffusers": importlib.metadata.version("diffusers"),
    "transformers": importlib.metadata.version("transformers"),
    "dtype": PROTOCOL.dtype,
    "max_sequence_length": PROTOCOL.max_sequence_length,
    "clip_skip": None,
    "prompt_routing": "same exact text to CLIP-L, CLIP-G, and T5-XXL",
    "full_t5": True,
    "encoding_batch_size": PROTOCOL.t5_cache_batch_size,
    "encoding_topology": "singleton_per_exact_text",
}
CACHE_IDENTITY_HASH = canonical_hash(CACHE_IDENTITY)


def embedding_key(text: str) -> str:
    return canonical_hash({"cache_identity": CACHE_IDENTITY, "text": text})


def embedding_path(text: str) -> Path:
    return EMBEDDING_ROOT / f"{embedding_key(text)}.safetensors"


required_texts = {PROTOCOL.negative_prompt}
for sample in rebuilt.samples:
    required_texts.add(sample.prompt)
    item = decomp_by_id[sample.id]
    required_texts.add(item.source_prompt)
    required_texts.update(p["text"] for p in item.primitive_prompts if p.get("enabled", True))
required_texts = sorted(required_texts)


def valid_embedding(text: str) -> bool:
    path = embedding_path(text)
    try:
        with safe_open(str(path), framework="pt", device="cpu") as handle:
            if set(handle.keys()) != {"pooled_prompt_embeds", "prompt_embeds"}:
                return False
            prompt_shape = tuple(handle.get_slice("prompt_embeds").get_shape())
            pooled_shape = tuple(handle.get_slice("pooled_prompt_embeds").get_shape())
            metadata = handle.metadata()
        return (
            prompt_shape == (1, 333, 4096)
            and pooled_shape == (1, 2048)
            and metadata.get("cache_identity_sha256") == CACHE_IDENTITY_HASH
            and metadata.get("entry_key") == embedding_key(text)
        )
    except Exception:
        return False


# A failed prior audit can leave the full pipeline referenced in the live kernel.
# Release only these cell-owned objects before validating/resuming the cache.
def release_full_text_stack():
    for stale_name in ("normal_cfg", "cached_positive", "cached_negative", "full_pipe", "tokenizer_3", "t5"):
        stale_object = globals().pop(stale_name, None)
        del stale_object
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()


release_full_text_stack()

previous_index = None
if EMBEDDING_INDEX.exists():
    try:
        previous_index = json.loads(EMBEDDING_INDEX.read_text(encoding="utf-8"))
        if previous_index.get("cache_identity_sha256") != CACHE_IDENTITY_HASH:
            previous_index = None
    except Exception:
        previous_index = None
pending = [text for text in required_texts if not valid_embedding(text)]
print(f"Full-T5 cache: {len(required_texts) - len(pending)}/{len(required_texts)} valid; {len(pending)} pending")
equivalence_audit = None if previous_index is None else previous_index.get("singleton_cfg_exact_equivalence_audit")
t5_device_map = None if previous_index is None else previous_index.get("t5_device_map")
needs_audit = not equivalence_audit or not equivalence_audit.get("passed", False)
if pending or needs_audit:
    # Jupyter fires post_run_cell after both successful and failed execution.
    # This one-shot guard releases a partially constructed sharded T5 stack on any exception.
    def _release_full_text_stack_after_cell(result):
        try:
            release_full_text_stack()
        finally:
            try:
                get_ipython().events.unregister("post_run_cell", _release_full_text_stack_after_cell)
            except ValueError:
                pass

    get_ipython().events.register("post_run_cell", _release_full_text_stack_after_cell)
    from diffusers import StableDiffusion3Pipeline
    from transformers import T5EncoderModel, T5TokenizerFast

    total_gib = [torch.cuda.get_device_properties(i).total_memory // (1024 ** 3) for i in range(2)]
    max_memory = {
        0: f"{max(4, int(total_gib[0]) - 2)}GiB",
        1: f"{max(4, int(total_gib[1]) - 2)}GiB",
        "cpu": "45GiB",
    }
    t5 = T5EncoderModel.from_pretrained(
        MODEL_SNAPSHOT / "text_encoder_3", torch_dtype=torch.float16,
        low_cpu_mem_usage=True, device_map="balanced", max_memory=max_memory,
    )
    t5_device_map = dict(t5.hf_device_map)
    tokenizer_3 = T5TokenizerFast.from_pretrained(MODEL_SNAPSHOT / "tokenizer_3")
    full_pipe = StableDiffusion3Pipeline.from_pretrained(
        str(MODEL_SNAPSHOT), torch_dtype=torch.float16, use_safetensors=True,
        low_cpu_mem_usage=True, text_encoder_3=t5, tokenizer_3=tokenizer_3,
    )
    full_pipe.set_progress_bar_config(disable=True)

    encode_common = dict(
        prompt_2=None, prompt_3=None, negative_prompt=None,
        negative_prompt_2=None, negative_prompt_3=None,
        do_classifier_free_guidance=False, device=torch.device("cpu"),
        num_images_per_prompt=1, max_sequence_length=PROTOCOL.max_sequence_length,
    )
    for start in tqdm(range(0, len(pending), PROTOCOL.t5_cache_batch_size), desc="Full SD3/T5 prompt encoding"):
        batch = pending[start:start + PROTOCOL.t5_cache_batch_size]
        with torch.inference_mode():
            encoded = full_pipe.encode_prompt(prompt=batch, **encode_common)
        prompt_batch, pooled_batch = encoded[0].cpu(), encoded[2].cpu()
        if prompt_batch.shape[1:] != (333, 4096) or pooled_batch.shape[1:] != (2048,):
            raise RuntimeError(f"Unexpected full-SD3 embedding shapes: {prompt_batch.shape}, {pooled_batch.shape}")
        for index, text_value in enumerate(batch):
            destination = embedding_path(text_value)
            temporary = destination.with_suffix(".part.safetensors")
            save_safetensors({
                "prompt_embeds": prompt_batch[index:index + 1].contiguous(),
                "pooled_prompt_embeds": pooled_batch[index:index + 1].contiguous(),
            }, str(temporary), metadata={
                "entry_key": embedding_key(text_value),
                "text_sha256": sha256_bytes(text_value.encode("utf-8")),
                "cache_identity_sha256": CACHE_IDENTITY_HASH,
                "model_revision": PROTOCOL.model_revision,
                "max_sequence_length": str(PROTOCOL.max_sequence_length),
                "text_encoders": ",".join(PROTOCOL.full_text_encoders),
            })
            roundtrip = load_safetensors(str(temporary), device="cpu")
            if not (
                torch.equal(roundtrip["prompt_embeds"], prompt_batch[index:index + 1])
                and torch.equal(roundtrip["pooled_prompt_embeds"], pooled_batch[index:index + 1])
            ):
                raise RuntimeError(f"Safetensors round-trip changed cached conditioning: {text_value!r}")
            del roundtrip
            os.replace(temporary, destination)
        del encoded, prompt_batch, pooled_batch

    # The cache uses singleton encoding, exactly matching per-prompt CFG topology.
    # Require bitwise equality for positive, negative, and pooled tensors.
    audit_prompt = rebuilt.samples[0].prompt
    with torch.inference_mode():
        normal_cfg = full_pipe.encode_prompt(
            prompt=audit_prompt, prompt_2=None, prompt_3=None,
            negative_prompt=PROTOCOL.negative_prompt, negative_prompt_2=None, negative_prompt_3=None,
            do_classifier_free_guidance=True, device=torch.device("cpu"),
            num_images_per_prompt=1, max_sequence_length=PROTOCOL.max_sequence_length,
        )
    cached_positive = load_safetensors(str(embedding_path(audit_prompt)), device="cpu")
    cached_negative = load_safetensors(str(embedding_path(PROTOCOL.negative_prompt)), device="cpu")
    def tensor_difference_stats(reference, candidate):
        reference_f32 = reference.float()
        candidate_f32 = candidate.float()
        difference = (reference_f32 - candidate_f32).abs()
        reference_norm = max(float(reference_f32.norm()), torch.finfo(torch.float32).tiny)
        reference_abs_max = max(float(reference_f32.abs().max()), torch.finfo(torch.float32).tiny)
        return {
            "exact": bool(torch.equal(reference, candidate)),
            "finite": bool(torch.isfinite(candidate).all()),
            "max_abs": float(difference.max()),
            "mean_abs": float(difference.mean()),
            "rmse": float(difference.square().mean().sqrt()),
            "relative_l2": float((reference_f32 - candidate_f32).norm()) / reference_norm,
            "max_abs_over_reference_max": float(difference.max()) / reference_abs_max,
        }

    positive_prompt_stats = tensor_difference_stats(normal_cfg[0], cached_positive["prompt_embeds"])
    positive_pooled_stats = tensor_difference_stats(normal_cfg[2], cached_positive["pooled_prompt_embeds"])
    negative_prompt_stats = tensor_difference_stats(normal_cfg[1], cached_negative["prompt_embeds"])
    negative_pooled_stats = tensor_difference_stats(normal_cfg[3], cached_negative["pooled_prompt_embeds"])
    audit_passed = (
        positive_prompt_stats["exact"] and negative_prompt_stats["exact"]
        and positive_pooled_stats["exact"] and negative_pooled_stats["exact"]
    )
    equivalence_audit = {
        "prompt": audit_prompt,
        "comparison": "cached singleton FP16 encode versus singleton FP16 CFG encode",
        "positive_prompt": positive_prompt_stats,
        "positive_pooled": positive_pooled_stats,
        "negative_prompt": negative_prompt_stats,
        "negative_pooled": negative_pooled_stats,
        "passed": bool(audit_passed),
    }
    audit_failure = None if audit_passed else RuntimeError(
        f"Cached singleton conditioning is not bitwise-identical to singleton CFG encoding: {equivalence_audit}"
    )
    release_full_text_stack()
    if audit_failure is not None:
        raise audit_failure

invalid = [text for text in required_texts if not valid_embedding(text)]
if invalid:
    raise RuntimeError(f"Full-T5 cache remains incomplete: {len(invalid)} invalid entries")
index_record = {
    "protocol_hash": PROTOCOL_HASH,
    "model_id": PROTOCOL.model_id, "model_revision": PROTOCOL.model_revision,
    "cache_identity": CACHE_IDENTITY,
    "cache_identity_sha256": CACHE_IDENTITY_HASH,
    "full_text_encoders": list(PROTOCOL.full_text_encoders),
    "max_sequence_length": PROTOCOL.max_sequence_length,
    "entry_count": len(required_texts),
    "entries": {
        text: {
            "key": embedding_key(text),
            "file": embedding_path(text).name,
            "sha256": sha256_file(embedding_path(text)),
            "prompt_shape": [1, 333, 4096],
            "pooled_shape": [1, 2048],
            "dtype": "torch.float16",
        }
        for text in required_texts
    },
    "t5_device_map": t5_device_map,
    "singleton_cfg_exact_equivalence_audit": equivalence_audit,
}
write_json(index_record, EMBEDDING_INDEX)
print(f"Validated {len(required_texts)} full-SD3/T5 conditioning entries ({EMBEDDING_INDEX})")


Full-T5 cache: 586/586 valid; 0 pending
Validated 586 full-SD3/T5 conditioning entries (/media/fezan/ASi/DVLM/steering/aim-flow/outputs/t2i_compbench_1000_seed13_full_t5_dual_gpu/full_t5_prompt_cache/index.json)


In [6]:
# Materialize a self-contained worker from this notebook. It uses repository implementations, not reimplemented guidance.
WORKER_SOURCE = r"""
from __future__ import annotations

import argparse
import gc
import hashlib
import json
import os
import random
import sys
import time
from pathlib import Path
from types import MethodType
from typing import Any

import numpy as np
import torch
from PIL import Image
from safetensors.torch import load_file as load_safetensors
from tqdm.auto import tqdm


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--repo-root", type=Path, required=True)
    parser.add_argument("--artifact-root", type=Path, required=True)
    parser.add_argument("--model-snapshot", type=Path, required=True)
    parser.add_argument("--rectified-repo", type=Path, required=True)
    parser.add_argument("--protocol", type=Path, required=True)
    parser.add_argument("--tasks", type=Path, required=True)
    parser.add_argument("--decompositions", type=Path, required=True)
    parser.add_argument("--embedding-index", type=Path, required=True)
    parser.add_argument("--physical-gpu", type=int, required=True)
    parser.add_argument("--methods", nargs="+", required=True)
    return parser.parse_args()


ARGS = parse_args()
sys.path.insert(0, str(ARGS.repo_root / "src"))
from aim_flow.eval_bench.generation import (
    RectifiedCFGPPBackend,
    _install_rectified_cfgpp_diffusers_compat,
    _validate_rectified_cfgpp_pipeline_import,
    load_bench_config,
    unload_model,
)
from aim_flow.prompt_schema import PrimitiveFlowSet
from aim_flow.sampler import AIMFlowSampler
from aim_flow.sd3_backend import SD3Backend, TextCondition

RECORD = json.loads(ARGS.protocol.read_text(encoding="utf-8"))
P = RECORD["protocol"]
PROTOCOL_HASH = RECORD["protocol_hash"]
TASKS = [json.loads(line) for line in ARGS.tasks.read_text(encoding="utf-8").splitlines() if line]
DECOMPOSITIONS = {item["id"]: item for item in json.loads(ARGS.decompositions.read_text(encoding="utf-8"))["items"]}
EMBEDDING_RECORD = json.loads(ARGS.embedding_index.read_text(encoding="utf-8"))
CACHE_IDENTITY = EMBEDDING_RECORD.get("cache_identity", {})
CACHE_AUDIT = EMBEDDING_RECORD.get("singleton_cfg_exact_equivalence_audit")
if int(P.get("t5_cache_batch_size", 0)) != 1:
    raise RuntimeError("Worker protocol is stale: singleton T5 cache topology is required")
if CACHE_IDENTITY.get("encoding_batch_size") != 1 or CACHE_IDENTITY.get("encoding_topology") != "singleton_per_exact_text":
    raise RuntimeError("Embedding index is stale: rebuild the singleton full-T5 cache cell")
if not CACHE_AUDIT or not CACHE_AUDIT.get("passed", False):
    raise RuntimeError("Embedding index lacks a passing singleton CFG exact-equivalence audit")
EMBEDDINGS = EMBEDDING_RECORD["entries"]
EMBEDDING_ROOT = ARGS.embedding_index.parent
VALID_METHODS = {"our_method", "base_cfg", "rectified_cfgpp"}
if not set(ARGS.methods) <= VALID_METHODS:
    raise ValueError(f"Unknown methods: {ARGS.methods}")
if not torch.cuda.is_available() or torch.cuda.device_count() != 1:
    raise RuntimeError("Each worker must see exactly one CUDA GPU through CUDA_VISIBLE_DEVICES")
torch.cuda.set_device(0)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
OVERWRITE_INVALID = os.environ.get("AIM_FLOW_OVERWRITE_INVALID", "0") == "1"


def seed_everything(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def sha256_file(path: Path, chunk_size: int = 1 << 20) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def canonical_hash(value: Any) -> str:
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":")).encode()).hexdigest()


def write_json(value: Any, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    os.replace(temporary, path)


def atomic_save_png(image: Image.Image, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp.png")
    image.save(temporary, format="PNG")
    with Image.open(temporary) as check:
        check.load()
        if check.size != (int(P["width"]), int(P["height"])):
            raise RuntimeError(f"Wrong generated image size: {check.size}")
    os.replace(temporary, path)


class EmbeddingStore:
    def __init__(self):
        self.cache = {}

    def get(self, text: str):
        if text not in EMBEDDINGS:
            raise KeyError(f"Text absent from full-T5 cache: {text!r}")
        if text not in self.cache:
            path = EMBEDDING_ROOT / EMBEDDINGS[text]["file"]
            expected_sha256 = EMBEDDINGS[text].get("sha256")
            if expected_sha256 and sha256_file(path) != expected_sha256:
                raise RuntimeError(f"Cached embedding checksum mismatch: {path}")
            tensors = load_safetensors(str(path), device="cpu")
            if (
                tuple(tensors["prompt_embeds"].shape) != (1, 333, 4096)
                or tuple(tensors["pooled_prompt_embeds"].shape) != (1, 2048)
                or tensors["prompt_embeds"].dtype != torch.float16
                or tensors["pooled_prompt_embeds"].dtype != torch.float16
            ):
                raise RuntimeError(f"Invalid cached prompt shape: {path}")
            self.cache[text] = tensors
        return self.cache[text]


STORE = EmbeddingStore()


def make_config(seed: int):
    cfg = load_bench_config(
        seed=seed,
        aggregation_steps_1_indexed=list(P["spfc_aggregation_steps_1_indexed"]),
        num_inference_steps=int(P["num_inference_steps"]),
        height=int(P["height"]), width=int(P["width"]),
        guidance_scale=float(P["guidance_scale"]),
    )
    cfg.model.model_id = str(ARGS.model_snapshot)
    cfg.model.dtype = P["dtype"]
    cfg.model.enable_model_cpu_offload = True
    cfg.model.load_t5_text_encoder = False
    cfg.model.enable_vae_slicing = True
    return cfg


def scheduler_record(pipe):
    config = dict(pipe.scheduler.config)
    # Diffusers stores the defaulted field names in a list whose insertion order
    # varies by pipeline construction path. It is provenance metadata, not scheduler state.
    semantic_config = dict(config)
    semantic_config.pop("_use_default_values", None)
    return {
        "class": f"{pipe.scheduler.__class__.__module__}.{pipe.scheduler.__class__.__name__}",
        "config": config,
        "semantic_config": semantic_config,
        "fingerprint": canonical_hash(semantic_config),
    }


def load_generator_only_pipeline(custom_pipeline: Path | None = None):
    # Full CLIP/T5 encoding finished before workers launch. Omitting all
    # text encoders here changes component placement only: each method
    # receives the exact cached four-tensor conditioning bundle.
    from diffusers import StableDiffusion3Pipeline

    kwargs = {
        "torch_dtype": torch.float16,
        "use_safetensors": True,
        "low_cpu_mem_usage": True,
        "text_encoder": None, "tokenizer": None,
        "text_encoder_2": None, "tokenizer_2": None,
        "text_encoder_3": None, "tokenizer_3": None,
    }
    if custom_pipeline is not None:
        kwargs["custom_pipeline"] = str(custom_pipeline)
    pipe = StableDiffusion3Pipeline.from_pretrained(str(ARGS.model_snapshot), **kwargs)
    pipe.enable_model_cpu_offload()
    if hasattr(pipe.vae, "enable_slicing"):
        pipe.vae.enable_slicing()
    pipe.set_progress_bar_config(disable=True)
    return pipe


def expected_metadata(method: str, task: dict[str, Any]):
    return {
        "benchmark": "t2i_compbench_seed13_subset",
        "method": method,
        "prompt": task["prompt"],
        "subset_sample_id": task["subset_sample_id"],
        "category": task["category"],
        "official_prompt_index": task["official_prompt_index"],
        "selected_category_rank": task["selected_category_rank"],
        "sample_index": task["sample_index"],
        "question_id": task["question_id"],
        "seed": task["seed"],
        "protocol_hash": PROTOCOL_HASH,
        "model_id": P["model_id"],
        "model_revision": P["model_revision"],
        "height": P["height"], "width": P["width"],
        "num_inference_steps": P["num_inference_steps"],
        "guidance_scale": P["guidance_scale"],
        "negative_prompt": P["negative_prompt"],
        "full_text_encoders": P["full_text_encoders"],
        "full_t5_conditioning": True,
        "physical_gpu": ARGS.physical_gpu,
    }


def output_paths(method: str, task: dict[str, Any]):
    stage = ARGS.artifact_root / "t2i_compbench" / method / task["category"]
    stem = f"{task['prompt']}_{int(task['question_id']):06d}"
    return stage / "samples" / f"{stem}.png", stage / "sample_metadata" / f"{int(task['question_id']):06d}.json"


def output_valid(image_path: Path, metadata_path: Path, expected: dict[str, Any]) -> bool:
    if not image_path.exists() and not metadata_path.exists():
        return False
    if image_path.exists() != metadata_path.exists():
        image_path.unlink(missing_ok=True)
        metadata_path.unlink(missing_ok=True)
        return False
    valid = False
    try:
        metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
        with Image.open(image_path) as image:
            image.verify()
        with Image.open(image_path) as image:
            valid = image.size == (int(P["width"]), int(P["height"]))
        valid = valid and all(metadata.get(key) == value for key, value in expected.items())
        valid = valid and metadata.get("image_sha256") == sha256_file(image_path)
    except Exception:
        valid = False
    if valid:
        return True
    if OVERWRITE_INVALID:
        image_path.unlink(missing_ok=True)
        metadata_path.unlink(missing_ok=True)
        return False
    raise RuntimeError(f"Existing output is corrupt or from another protocol: {image_path}")


def cached_condition(self, prompt: str, negative_prompt=None, device=None, do_classifier_free_guidance=False):
    if (negative_prompt or "") != P["negative_prompt"]:
        raise RuntimeError("Only the locked shared empty negative prompt is allowed")
    positive = STORE.get(prompt)
    negative = STORE.get(P["negative_prompt"]) if do_classifier_free_guidance else None
    condition = TextCondition(
        prompt=prompt,
        prompt_embeds=positive["prompt_embeds"],
        pooled_prompt_embeds=positive["pooled_prompt_embeds"],
        negative_prompt_embeds=None if negative is None else negative["prompt_embeds"],
        negative_pooled_prompt_embeds=None if negative is None else negative["pooled_prompt_embeds"],
    )
    condition.validate()
    return condition.to(device or self.execution_device, self.dtype)


def pipeline_embeddings(prompt: str, backend):
    positive = STORE.get(prompt)
    negative = STORE.get(P["negative_prompt"])
    device = backend.execution_device if hasattr(backend, "execution_device") else torch.device("cuda:0")
    dtype = backend.dtype
    return {
        "prompt_embeds": positive["prompt_embeds"].to(device=device, dtype=dtype),
        "pooled_prompt_embeds": positive["pooled_prompt_embeds"].to(device=device, dtype=dtype),
        "negative_prompt_embeds": negative["prompt_embeds"].to(device=device, dtype=dtype),
        "negative_pooled_prompt_embeds": negative["pooled_prompt_embeds"].to(device=device, dtype=dtype),
    }


def run_method(method: str):
    seed_everything(int(P["generation_seed_start"]))
    cfg = make_config(int(P["generation_seed_start"]))
    if method in {"our_method", "base_cfg"}:
        backend = SD3Backend(cfg)
        backend.pipe = load_generator_only_pipeline()
        sampler = AIMFlowSampler(backend, cfg) if method == "our_method" else None
        if method == "our_method":
            backend.encode_text_condition = MethodType(cached_condition, backend)
    else:
        backend = RectifiedCFGPPBackend(
            cfg, repo_dir=ARGS.rectified_repo,
            sigma_noise=float(P["rectified_sigma_noise"]),
        )
        _install_rectified_cfgpp_diffusers_compat()
        _validate_rectified_cfgpp_pipeline_import(ARGS.rectified_repo)
        backend.pipe = load_generator_only_pipeline(ARGS.rectified_repo / "rect-cfg-SD3-pipeline")
        sampler = None
    scheduler = scheduler_record(backend.pipe)
    if "FlowMatchEulerDiscreteScheduler" not in scheduler["class"]:
        raise RuntimeError(f"Unexpected scheduler: {scheduler['class']}")
    completed = 0
    started_method = time.perf_counter()
    try:
        for task in tqdm(TASKS, desc=f"GPU{ARGS.physical_gpu}/{method}", unit="image"):
            image_path, metadata_path = output_paths(method, task)
            expected = expected_metadata(method, task)
            if output_valid(image_path, metadata_path, expected):
                completed += 1
                continue
            seed_everything(int(task["seed"]))
            cfg.sampler.seed = int(task["seed"])
            started = time.perf_counter()
            if method == "our_method":
                item = DECOMPOSITIONS[task["subset_sample_id"]]
                flow_set = PrimitiveFlowSet.from_dict({
                    "name": item["id"], "target_prompt": item["target_prompt"],
                    "source_prompt": item["source_prompt"],
                    "primitive_prompts": item["primitive_prompts"],
                    "negative_prompt": P["negative_prompt"],
                })
                image, method_metadata = sampler.generate_sparse_primitive_flow(flow_set, mode="primitive_flow_sparse")
            else:
                embeds = pipeline_embeddings(task["prompt"], backend)
                generator = torch.Generator(device="cpu").manual_seed(int(task["seed"]))
                common = dict(
                    prompt=None, negative_prompt=None,
                    height=int(P["height"]), width=int(P["width"]),
                    num_inference_steps=int(P["num_inference_steps"]),
                    num_images_per_prompt=1, generator=generator, **embeds,
                )
                if method == "base_cfg":
                    result = backend.pipe(guidance_scale=float(P["guidance_scale"]), **common)
                else:
                    result = backend.pipe(
                        true_cfg=float(P["guidance_scale"]),
                        sigma_noise=float(P["rectified_sigma_noise"]), **common,
                    )
                image, method_metadata = result.images[0], None
                del embeds, result
            runtime_seconds = time.perf_counter() - started
            atomic_save_png(image, image_path)
            metadata = {
                **expected,
                "runtime_seconds": runtime_seconds,
                "image_sha256": sha256_file(image_path),
                "scheduler": scheduler,
                "embedding_key_positive": EMBEDDINGS[task["prompt"]]["key"],
                "embedding_key_negative": EMBEDDINGS[P["negative_prompt"]]["key"],
                "rectified_cfgpp_commit": RECORD["repository_pins"]["rectified_cfgpp"]["commit"] if method == "rectified_cfgpp" else None,
                "rectified_sigma_noise": P["rectified_sigma_noise"] if method == "rectified_cfgpp" else None,
                "method_metadata": method_metadata,
            }
            write_json(metadata, metadata_path)
            completed += 1
            del image, method_metadata
            torch.cuda.empty_cache()
    finally:
        write_json({
            "method": method, "physical_gpu": ARGS.physical_gpu,
            "completed": completed, "expected": len(TASKS),
            "elapsed_seconds": time.perf_counter() - started_method,
            "scheduler": scheduler, "protocol_hash": PROTOCOL_HASH,
        }, ARGS.artifact_root / "t2i_compbench" / method / "generation_run.json")
        del sampler
        unload_model(backend)
        del backend
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


for requested_method in ARGS.methods:
    run_method(requested_method)
"""

RUNTIME_DIR = ARTIFACT_ROOT / "runtime"
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
WORKER_PATH = RUNTIME_DIR / "dual_gpu_generation_worker.py"
WORKER_PATH.write_text(WORKER_SOURCE.strip() + "\n", encoding="utf-8")
print(f"Worker: {WORKER_PATH}\nSHA256: {sha256_file(WORKER_PATH)}")


Worker: /media/fezan/ASi/DVLM/steering/aim-flow/outputs/t2i_compbench_1000_seed13_full_t5_dual_gpu/runtime/dual_gpu_generation_worker.py
SHA256: fa48963aa0733859f6718eb88426719968ca8666a2cf53a0cf95a60316a60bf1


In [7]:
# # Launch the balanced dual-GPU workload. Rerunning resumes valid samples.
# import subprocess

# ASSIGNMENTS = {
#     PROTOCOL.physical_gpu_our_method: ["our_method"],
#     PROTOCOL.physical_gpu_baselines: ["base_cfg", "rectified_cfgpp"],
# }
# WORKER_LOG_DIR = ARTIFACT_ROOT / "logs" / "generation"
# WORKER_LOG_DIR.mkdir(parents=True, exist_ok=True)
# processes = {}
# log_handles = {}
# for physical_gpu, assigned_methods in ASSIGNMENTS.items():
#     env = os.environ.copy()
#     env["CUDA_VISIBLE_DEVICES"] = str(physical_gpu)
#     env["PYTHONUNBUFFERED"] = "1"
#     env["PYTHONHASHSEED"] = "0"
#     env["PYTHONPATH"] = str(REPO_ROOT / "src") + os.pathsep + env.get("PYTHONPATH", "")
#     command = [
#         sys.executable, str(WORKER_PATH),
#         "--repo-root", str(REPO_ROOT),
#         "--artifact-root", str(ARTIFACT_ROOT),
#         "--model-snapshot", str(MODEL_SNAPSHOT),
#         "--rectified-repo", str(REPO_PATHS["rectified_cfgpp"]),
#         "--protocol", str(ARTIFACT_ROOT / "protocol.json"),
#         "--tasks", str(TASK_MANIFEST),
#         "--decompositions", str(MANIFEST_DIR / "spfc_decompositions.json"),
#         "--embedding-index", str(EMBEDDING_INDEX),
#         "--physical-gpu", str(physical_gpu),
#         "--methods", *assigned_methods,
#     ]
#     log_path = WORKER_LOG_DIR / f"gpu_{physical_gpu}.log"
#     handle = log_path.open("a", encoding="utf-8")
#     handle.write("\n+ " + " ".join(command) + "\n")
#     handle.flush()
#     processes[physical_gpu] = subprocess.Popen(command, env=env, stdout=handle, stderr=subprocess.STDOUT, text=True)
#     log_handles[physical_gpu] = handle
#     print(f"GPU {physical_gpu}: {assigned_methods}; PID={processes[physical_gpu].pid}; log={log_path}")


# def generation_counts():
#     return {
#         method: sum(1 for _ in (ARTIFACT_ROOT / "t2i_compbench" / method).glob("*/samples/*.png"))
#         for method in METHODS
#     }


# try:
#     while any(process.poll() is None for process in processes.values()):
#         print(time.strftime("%Y-%m-%d %H:%M:%S"), generation_counts(), {gpu: p.poll() for gpu, p in processes.items()})
#         time.sleep(15)
# except KeyboardInterrupt:
#     for process in processes.values():
#         if process.poll() is None:
#             process.terminate()
#     raise
# finally:
#     for handle in log_handles.values():
#         handle.close()
# failures = {gpu: process.returncode for gpu, process in processes.items() if process.returncode != 0}
# if failures:
#     raise RuntimeError(f"Generation worker failure(s): {failures}. Inspect {WORKER_LOG_DIR}")
# print("Generation workers completed:", generation_counts())


In [8]:
# Validate exact counts, image/metadata pairing, seeds, prompts, and shared scheduler configuration.
generation_audit = {"protocol_hash": PROTOCOL_HASH, "methods": {}}
scheduler_fingerprints = {}
scheduler_classes = {}
task_keys = {(task["category"], task["question_id"]): task for task in tasks}
for method in METHODS:
    method_record = {"categories": {}}
    seen = set()
    for category in T2I_CATEGORIES:
        stage = ARTIFACT_ROOT / "t2i_compbench" / method / category
        images = sorted((stage / "samples").glob("*.png"))
        sidecars = sorted((stage / "sample_metadata").glob("*.json"))
        if len(images) != 250 or len(sidecars) != 250:
            raise RuntimeError(f"Expected 250 image/metadata pairs for {method}/{category}; got {len(images)}/{len(sidecars)}")
        image_by_qid = {int(path.stem.rsplit("_", 1)[1]): path for path in images}
        metadata_by_qid = {int(path.stem): path for path in sidecars}
        if set(image_by_qid) != set(metadata_by_qid):
            raise RuntimeError(f"Image/metadata question IDs differ for {method}/{category}")
        category_hash_rows = []
        for question_id, image_path in image_by_qid.items():
            task = task_keys[(category, question_id)]
            metadata = json.loads(metadata_by_qid[question_id].read_text(encoding="utf-8"))
            checks = {
                "prompt": task["prompt"], "seed": task["seed"],
                "sample_index": task["sample_index"], "protocol_hash": PROTOCOL_HASH,
                "method": method, "full_t5_conditioning": True,
            }
            if any(metadata.get(key) != value for key, value in checks.items()):
                raise RuntimeError(f"Sidecar mismatch: {metadata_by_qid[question_id]}")
            if metadata["image_sha256"] != sha256_file(image_path):
                raise RuntimeError(f"Image checksum mismatch: {image_path}")
            seen.add((category, question_id))
            category_hash_rows.append((image_path.name, metadata["image_sha256"]))
        method_record["categories"][category] = {
            "count": len(images),
            "image_set_sha256": canonical_hash(sorted(category_hash_rows)),
        }
    if seen != set(task_keys):
        raise RuntimeError(f"Method task coverage mismatch: {method}")
    run_record = json.loads((ARTIFACT_ROOT / "t2i_compbench" / method / "generation_run.json").read_text())
    scheduler_config = dict(run_record["scheduler"]["config"])
    scheduler_config.pop("_use_default_values", None)
    scheduler_fingerprints[method] = canonical_hash(scheduler_config)
    scheduler_classes[method] = run_record["scheduler"]["class"]
    method_record["scheduler"] = run_record["scheduler"]
    method_record["total"] = len(seen)
    generation_audit["methods"][method] = method_record
if len(set(scheduler_fingerprints.values())) != 1:
    raise RuntimeError(f"Scheduler configurations differ across methods: {scheduler_fingerprints}")
if len(set(scheduler_classes.values())) != 1:
    raise RuntimeError(f"Scheduler classes differ across methods: {scheduler_classes}")
generation_audit["shared_scheduler_fingerprint"] = next(iter(scheduler_fingerprints.values()))
write_json(generation_audit, ARTIFACT_ROOT / "t2i_compbench" / "generation_audit.json")
print(json.dumps({m: generation_audit["methods"][m]["total"] for m in METHODS}, indent=2))


{
  "our_method": 1000,
  "base_cfg": 1000,
  "rectified_cfgpp": 1000
}


## Official T2I-CompBench scoring

The next cells install and execute the repository's official metric code without reimplementing the metrics: BLIP-VQA for color, texture, and shape; UniDet RS200 for 2D spatial relationships. Each category contains 25 selected prompts × 10 images = 250 scorer inputs per method.

**Documented installation-only deviation:** the official requirements list Detectron2 before PyTorch, but the pinned Detectron2 `setup.py` imports PyTorch while pip is still preparing metadata. The environment therefore installs the same official PyTorch/Torchvision pins first, installs the remaining official lock, and builds the exact pinned Detectron2 commit separately with CUDA 11.7. This changes neither benchmark code nor metric behavior; the post-install probe verifies CUDA, the compiled Detectron2 extension, spaCy model, package versions, and repository commit before scoring.


In [9]:
# Create/reuse the pinned isolated official evaluator environment and verify the UniDet checkpoint.
T2I_ENV = REPO_ROOT / ".venv" / "t2i-compbench-py310"
T2I_ENV_PYTHON = T2I_ENV / "bin" / "python"
T2I_LOCK_DIR = ARTIFACT_ROOT / "environment_locks"
T2I_LOCK_DIR.mkdir(parents=True, exist_ok=True)
T2I_REQUIREMENTS_LOCK = T2I_LOCK_DIR / "t2i_compbench_requirements_pinned.txt"
T2I_STAGED_REQUIREMENTS_LOCK = T2I_LOCK_DIR / "t2i_compbench_requirements_staged.txt"
T2I_TORCH_BOOTSTRAP = ("torch==2.0.1", "torchvision==0.15.2")
T2I_DETECTRON2_COMMIT = PINS.get("detectron2", (None, "5aeb252b194b93dc2879b4ac34bc51a31b5aee13"))[1]
T2I_ENV_BUILDER_SCHEMA = "official_t2i_cuda117_staged_v2"


def conda_executable() -> str:
    executable = shutil.which("conda")
    if not executable:
        raise FileNotFoundError("conda is required for the official T2I-CompBench evaluator environment")
    return executable


def ensure_t2i_environment():
    if not T2I_ENV_PYTHON.exists():
        run([conda_executable(), "create", "-y", "-p", str(T2I_ENV), "python=3.10", "pip=23.2.1"])
    build_env = os.environ.copy()
    build_env["CUDA_HOME"] = str(T2I_ENV)
    build_env["PATH"] = str(T2I_ENV / "bin") + os.pathsep + build_env.get("PATH", "")
    build_env["LD_LIBRARY_PATH"] = str(T2I_ENV / "lib") + os.pathsep + build_env.get("LD_LIBRARY_PATH", "")
    build_env["CC"] = str(T2I_ENV / "bin" / "x86_64-conda-linux-gnu-gcc")
    build_env["CXX"] = str(T2I_ENV / "bin" / "x86_64-conda-linux-gnu-g++")
    build_env["CPATH"] = str(T2I_ENV / "include") + os.pathsep + build_env.get("CPATH", "")
    build_env["CFLAGS"] = f"-I{T2I_ENV / 'include'}"
    build_env["CXXFLAGS"] = f"-I{T2I_ENV / 'include'}"
    build_env["TORCH_CUDA_ARCH_LIST"] = ";".join(sorted({
        f"{major}.{minor}" for major, minor in
        (torch.cuda.get_device_capability(i) for i in range(torch.cuda.device_count()))
    }))
    build_env["FORCE_CUDA"] = "1"
    build_env["MAX_JOBS"] = "2"
    official = (T2I_REPO / "requirements.txt").read_text(encoding="utf-8")
    official = official.replace(
        "git+https://github.com/openai/CLIP.git",
        f"git+https://github.com/openai/CLIP.git@{PINS['openai_clip'][1]}",
    )
    T2I_REQUIREMENTS_LOCK.write_text(official, encoding="utf-8")
    staged_lines = [
        line for line in official.splitlines()
        if not line.startswith("git+https://github.com/facebookresearch/detectron2.git")
        and not line.startswith("torch==")
        and not line.startswith("torchvision==")
    ]
    T2I_STAGED_REQUIREMENTS_LOCK.write_text("\n".join(staged_lines) + "\n", encoding="utf-8")
    marker = T2I_ENV / ".requirements_sha256"
    lock_hash = canonical_hash({
        "official_requirements_sha256": sha256_file(T2I_REQUIREMENTS_LOCK),
        "builder_schema": T2I_ENV_BUILDER_SCHEMA,
        "detectron2_commit": T2I_DETECTRON2_COMMIT,
        "torch_bootstrap": T2I_TORCH_BOOTSTRAP,
    })
    probe_code = (
        "import importlib.metadata as md, json, torch, torchvision, transformers, detectron2, detectron2._C, spacy; "
        "assert torch.cuda.is_available(); assert torch.__version__.startswith('2.0.1'); "
        "assert torchvision.__version__.startswith('0.15.2'); assert transformers.__version__ == '4.30.2'; "
        "assert detectron2._C.has_cuda(); spacy.load('en_core_web_sm'); "
        "d=json.loads(md.distribution('detectron2').read_text('direct_url.json')); "
        f"assert d['vcs_info']['commit_id'] == '{T2I_DETECTRON2_COMMIT}'; "
        "print(torch.__version__, torch.version.cuda, transformers.__version__, "
        "detectron2._C.get_cuda_version(), d['vcs_info']['commit_id'])"
    )
    probe = subprocess.run(
        [str(T2I_ENV_PYTHON), "-c", probe_code], cwd=str(REPO_ROOT), env=build_env,
        text=True, capture_output=True,
    )
    if probe.returncode != 0:
        # Detectron2 imports torch while preparing its package metadata; install
        # the official CUDA 11.7 torch pins before any source build.
        run([str(T2I_ENV_PYTHON), "-m", "pip", "install", "--no-cache-dir",
             *T2I_TORCH_BOOTSTRAP, "--index-url", "https://download.pytorch.org/whl/cu117"])
        run([str(T2I_ENV_PYTHON), "-m", "pip", "install", "setuptools<70", "ninja==1.13.0"])
        run([conda_executable(), "install", "-y", "-p", str(T2I_ENV),
             "gcc_linux-64=11", "gxx_linux-64=11"])
        run([conda_executable(), "install", "-y", "-p", str(T2I_ENV),
             "-c", "nvidia/label/cuda-11.7.1", "-c", "nvidia",
             "cuda-nvcc=11.7.99", "cuda-cccl=11.7.91",
             "cuda-cudart=11.7.99", "cuda-cudart-dev=11.7.99",
             "cuda-driver-dev=11.7.99", "cuda-nvrtc=11.7.99", "cuda-nvrtc-dev=11.7.99",
             "libcublas=11.10.3.66", "libcublas-dev=11.10.3.66",
             "libcusparse=11.7.4.91", "libcusparse-dev=11.7.4.91",
             "libcusolver=11.4.0.1", "libcusolver-dev=11.4.0.1", "cuda-version=11.7"])
        run([str(T2I_ENV_PYTHON), "-m", "pip", "install", "--no-cache-dir",
             "-r", str(T2I_STAGED_REQUIREMENTS_LOCK)], env=build_env)
        run([str(T2I_ENV_PYTHON), "-m", "pip", "install", "--no-cache-dir",
             "--no-build-isolation", "--no-deps",
             f"git+https://github.com/facebookresearch/detectron2.git@{T2I_DETECTRON2_COMMIT}"], env=build_env)
    else:
        print("Pinned CUDA evaluator environment already passes the strict probe; skipping rebuild.")
    check = run([str(T2I_ENV_PYTHON), "-c", probe_code], env=build_env, capture=True)
    print(check.stdout.strip())
    if not marker.exists() or marker.read_text().strip() != lock_hash:
        marker.write_text(lock_hash + "\n", encoding="utf-8")
    frozen = run([str(T2I_ENV_PYTHON), "-m", "pip", "freeze", "--all"], capture=True)
    (T2I_LOCK_DIR / "t2i_compbench_pip_freeze.txt").write_text(frozen.stdout, encoding="utf-8")


ensure_t2i_environment()

UNIDET_WEIGHT = T2I_REPO / "UniDet_eval" / "experts" / "expert_weights" / "Unified_learned_OCIM_RS200_6x+2x.pth"
UNIDET_URL = "https://huggingface.co/shikunl/prismer/resolve/main/expert_weights/Unified_learned_OCIM_RS200_6x%2B2x.pth"
UNIDET_SHA256 = "b19a836811c0d0d7aebf04f83ff4ea1da77ccf57b85cd65819746c1fccbff258"


def download_verified(url: str, destination: Path, expected_sha256: str):
    destination.parent.mkdir(parents=True, exist_ok=True)
    if not destination.exists():
        temporary = destination.with_suffix(destination.suffix + ".part")
        urllib.request.urlretrieve(url, temporary)
        os.replace(temporary, destination)
    actual = sha256_file(destination)
    if actual != expected_sha256:
        raise RuntimeError(f"Checkpoint checksum mismatch for {destination}: {actual}")
    return destination


download_verified(UNIDET_URL, UNIDET_WEIGHT, UNIDET_SHA256)
print("Official scorer environment and UniDet checkpoint: READY")


Pinned CUDA evaluator environment already passes the strict probe; skipping rebuild.
+ /media/fezan/ASi/DVLM/steering/aim-flow/.venv/t2i-compbench-py310/bin/python -c import importlib.metadata as md, json, torch, torchvision, transformers, detectron2, detectron2._C, spacy; assert torch.cuda.is_available(); assert torch.__version__.startswith('2.0.1'); assert torchvision.__version__.startswith('0.15.2'); assert transformers.__version__ == '4.30.2'; assert detectron2._C.has_cuda(); spacy.load('en_core_web_sm'); d=json.loads(md.distribution('detectron2').read_text('direct_url.json')); assert d['vcs_info']['commit_id'] == '5aeb252b194b93dc2879b4ac34bc51a31b5aee13'; print(torch.__version__, torch.version.cuda, transformers.__version__, detectron2._C.get_cuda_version(), d['vcs_info']['commit_id'])
2.0.1+cu117 11.7 4.30.2 CUDA 11.7 5aeb252b194b93dc2879b4ac34bc51a31b5aee13
+ /media/fezan/ASi/DVLM/steering/aim-flow/.venv/t2i-compbench-py310/bin/python -m pip freeze --all
Official scorer environ

In [10]:
# Run the official scorers, retain raw outputs/logs, and aggregate the four subset scores.
def evaluator_env():
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = str(PROTOCOL.physical_gpu_our_method)
    env["PYTHONUNBUFFERED"] = "1"
    return env


def run_logged(command: list[str], cwd: Path, log_path: Path):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print("+", " ".join(command), f"(log: {log_path})")
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(command, cwd=str(cwd), env=evaluator_env(), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        assert process.stdout is not None
        for line in process.stdout:
            log.write(line)
            if "score" in line.lower() or "processed" in line.lower():
                print(line.rstrip())
        return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)


def average_answers(path: Path, expected_count: int = 250) -> float:
    data = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(data, list) or len(data) != expected_count:
        raise RuntimeError(f"Expected {expected_count} official records in {path}; found {len(data) if isinstance(data, list) else type(data)}")
    values = [float(item["answer"]) for item in data]
    if not all(math.isfinite(value) for value in values):
        raise RuntimeError(f"Non-finite official scorer output in {path}")
    return float(np.mean(values))


def category_image_digest(method: str, category: str) -> str:
    stage = ARTIFACT_ROOT / "t2i_compbench" / method / category
    rows = []
    for sidecar in sorted((stage / "sample_metadata").glob("*.json")):
        metadata = json.loads(sidecar.read_text(encoding="utf-8"))
        rows.append((metadata["question_id"], metadata["image_sha256"]))
    if len(rows) != 250:
        raise RuntimeError(f"Generation incomplete: {method}/{category}")
    return canonical_hash(rows)


def run_official_t2i_scoring():
    scores = {}
    for method in METHODS:
        method_scores = {}
        for category in T2I_CATEGORIES:
            stage = ARTIFACT_ROOT / "t2i_compbench" / method / category
            raw_dir = stage / "raw_scorer_outputs"
            raw_dir.mkdir(parents=True, exist_ok=True)
            provenance = {
                "protocol_hash": PROTOCOL_HASH,
                "official_repo_commit": PINS["t2i_compbench"][1],
                "method": method, "category": category,
                "expected_count": 250,
                "image_set_sha256": category_image_digest(method, category),
            }
            provenance_path = raw_dir / "provenance.json"
            if category == "spatial":
                command = [str(T2I_ENV_PYTHON), "2D_spatial_eval.py", "--outpath", str(stage)]
                cwd = T2I_REPO / "UniDet_eval"
                result_path = stage / "labels" / "annotation_obj_detection_2d" / "vqa_result.json"
            else:
                command = [str(T2I_ENV_PYTHON), "BLIP_vqa.py", "--out_dir", str(stage), "--np_num", "8"]
                cwd = T2I_REPO / "BLIPvqa_eval"
                result_path = stage / "annotation_blip" / "vqa_result.json"
            resume = False
            if provenance_path.exists() and result_path.exists():
                resume = json.loads(provenance_path.read_text()) == provenance
                if resume:
                    average_answers(result_path)
            if not resume:
                run_logged(command, cwd, raw_dir / "official_scorer.log")
                write_json(provenance, provenance_path)
            score = average_answers(result_path)
            shutil.copy2(result_path, raw_dir / "vqa_result.json")
            method_scores[category] = score
            write_json({**provenance, "score": score, "raw_result": str(result_path)}, raw_dir / "score.json")
            gc.collect()
            torch.cuda.empty_cache()
        method_scores["aggregate"] = float(np.mean([method_scores[category] for category in T2I_CATEGORIES]))
        scores[method] = method_scores
        write_json(method_scores, ARTIFACT_ROOT / "t2i_compbench" / method / "final_results.json")
    write_json({"protocol_hash": PROTOCOL_HASH, "subset": "100 prompts / 1,000 images per method", "scores": scores}, ARTIFACT_ROOT / "t2i_compbench" / "final_results.json")
    return scores


T2I_SCORES = run_official_t2i_scoring()
pd.DataFrame(T2I_SCORES).T


+ /media/fezan/ASi/DVLM/steering/aim-flow/.venv/t2i-compbench-py310/bin/python BLIP_vqa.py --out_dir /media/fezan/ASi/DVLM/steering/aim-flow/outputs/t2i_compbench_1000_seed13_full_t5_dual_gpu/t2i_compbench/our_method/color --np_num 8 (log: /media/fezan/ASi/DVLM/steering/aim-flow/outputs/t2i_compbench_1000_seed13_full_t5_dual_gpu/t2i_compbench/our_method/color/raw_scorer_outputs/official_scorer.log)
Number of Processed Images: 250
Number of Processed Images: 250
Number of Processed Images: 250
Number of Processed Images: 250
Number of Processed Images: 250
Number of Processed Images: 250
Number of Processed Images: 250
Number of Processed Images: 250
BLIP-VQA score: 0.7436816000000003 !
+ /media/fezan/ASi/DVLM/steering/aim-flow/.venv/t2i-compbench-py310/bin/python BLIP_vqa.py --out_dir /media/fezan/ASi/DVLM/steering/aim-flow/outputs/t2i_compbench_1000_seed13_full_t5_dual_gpu/t2i_compbench/our_method/texture --np_num 8 (log: /media/fezan/ASi/DVLM/steering/aim-flow/outputs/t2i_compbench_1

,color,texture,spatial,shape,aggregate
our_method,0.743682,0.602500,0.217031,0.515322,0.519634
base_cfg,0.790947,0.670652,0.243990,0.558189,0.565945
rectified_cfgpp,0.781243,0.639810,0.216598,0.575086,0.553184


## Final seed-13 subset comparison and reproducibility export

All values below are official T2I-CompBench metric outputs on the same selected 100 prompts and 1,000 images per method. Higher is better for every column.


In [11]:
rows = []
for method in METHODS:
    score = T2I_SCORES[method]
    rows.append({
        "Method": METHOD_LABELS[method],
        "T2I Color": score["color"],
        "T2I Texture": score["texture"],
        "T2I Spatial": score["spatial"],
        "T2I Shape": score["shape"],
        "T2I Aggregate": score["aggregate"],
    })
FINAL_RESULTS = pd.DataFrame(rows)
RESULTS_DIR = ARTIFACT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FINAL_RESULTS.to_csv(RESULTS_DIR / "t2i_compbench_seed13_subset_results.csv", index=False)
write_json(rows, RESULTS_DIR / "t2i_compbench_seed13_subset_results.json")

package_names = [
    "torch", "torchvision", "diffusers", "transformers", "accelerate",
    "huggingface-hub", "safetensors", "sentencepiece", "protobuf",
    "numpy", "Pillow", "pandas", "tqdm",
]
package_versions = {}
for name in package_names:
    try:
        package_versions[name] = importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        package_versions[name] = None
reproducibility = {
    **protocol_record,
    "subset_audit": subset_audit,
    "full_t5_cache_index": str(EMBEDDING_INDEX),
    "full_t5_cache_index_sha256": sha256_file(EMBEDDING_INDEX),
    "worker_source": str(WORKER_PATH),
    "worker_source_sha256": sha256_file(WORKER_PATH),
    "generation_audit": generation_audit,
    "generation_package_versions": package_versions,
    "official_scorer_freeze": str(T2I_LOCK_DIR / "t2i_compbench_pip_freeze.txt"),
    "official_scorer_freeze_sha256": sha256_file(T2I_LOCK_DIR / "t2i_compbench_pip_freeze.txt"),
    "deviation": "Requested deterministic 25-per-category subset; not the full official 300-per-category benchmark protocol.",
    "result_files": {
        "csv": str(RESULTS_DIR / "t2i_compbench_seed13_subset_results.csv"),
        "json": str(RESULTS_DIR / "t2i_compbench_seed13_subset_results.json"),
    },
}
write_json(reproducibility, RESULTS_DIR / "reproducibility_manifest.json")
display(FINAL_RESULTS.style.format({column: "{:.6f}" for column in FINAL_RESULTS.columns if column != "Method"}))
print(f"CSV:  {RESULTS_DIR / 't2i_compbench_seed13_subset_results.csv'}")
print(f"JSON: {RESULTS_DIR / 't2i_compbench_seed13_subset_results.json'}")


,Method,T2I Color,T2I Texture,T2I Spatial,T2I Shape,T2I Aggregate
0,Our Method,0.743682,0.602500,0.217031,0.515322,0.519634
1,Base CFG,0.790947,0.670652,0.243990,0.558189,0.565945
2,Rectified CFG++,0.781243,0.639810,0.216598,0.575086,0.553184


CSV:  /media/fezan/ASi/DVLM/steering/aim-flow/outputs/t2i_compbench_1000_seed13_full_t5_dual_gpu/results/t2i_compbench_seed13_subset_results.csv
JSON: /media/fezan/ASi/DVLM/steering/aim-flow/outputs/t2i_compbench_1000_seed13_full_t5_dual_gpu/results/t2i_compbench_seed13_subset_results.json
